In [1]:
import torch
import torch.nn as nn
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm
import random
import numpy as np # numpy 임포트
from datetime import datetime

# --- 0. 랜덤 시드 고정 ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 멀티 GPU를 위한 설정
    torch.backends.cudnn.deterministic = True # 결정론적 CUDNN 동작
    torch.backends.cudnn.benchmark = False # 벤치마크 모드 비활성화 (성능 저하 가능성, 재현성 위함)

SEED = 42 # 원하는 시드 값
set_seed(SEED)
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 모든 랜덤 시드를 {SEED}로 고정했습니다.")


/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-07-03 15:53:33] 모든 랜덤 시드를 42로 고정했습니다.


In [2]:
# --- 1. 설정 및 경로 정의 ---
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 설정 시작...")

# 모델 이름 (대회 규칙: 2024년 이전 공개, 오픈소스, 총 파라미터 3B 미만)
CLIP_MODEL_NAME = "openai/clip-vit-large-patch14" # 약 0.3B 파라미터
GPT2_MODEL_NAME = "gpt2-xl"                       # 약 1.558B 파라미터
# 총 파라미터: 0.3B + 1.558B = 1.858B. 매핑 네트워크 파라미터가 1.142B 미만이면 OK.

# 절대 경로 (VMCBench 데이터셋 위치)
VMCBENCH_BASE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/clip_gpt/VMCBench_from_CSV_as_Official_VQAv2"

# 훈련 데이터 경로
TRAIN_ANNOTATION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_mscoco_train2014_annotations.json")
TRAIN_QUESTION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_OpenEnded_mscoco_train2014_questions.json")
TRAIN_IMAGE_DIR = os.path.join(VMCBENCH_BASE_DIR, "train2014")

# 검증 데이터 경로
VAL_ANNOTATION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_mscoco_val2014_annotations.json")
VAL_QUESTION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_OpenEnded_mscoco_val2014_questions.json")
VAL_IMAGE_DIR = os.path.join(VMCBENCH_BASE_DIR, "val2014")

# 하이퍼파라미터
BATCH_SIZE = 2 # GPU 메모리 상황에 따라 조절. GPT-2 XL은 매우 큽니다.
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4 # 오버피팅 시 5e-5 또는 1e-5로 줄여볼 것
NUM_SOFT_TOKENS = 20 # 가상 토큰 개수. 오버피팅 시 줄여볼 것 (예: 10)

# 장치 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용할 장치: {device}")

[2025-07-03 15:53:33] 설정 시작...
사용할 장치: cuda


In [3]:
# --- 2. 모델 및 토크나이저 로드 ---
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 모델 및 토크나이저 로드 중...")

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
clip_model.eval() # Freeze CLIP
for param in clip_model.parameters():
    param.requires_grad = False
print(f"CLIP 모델 로드 완료. 파라미터 수: {sum(p.numel() for p in clip_model.parameters()) / 1e9:.3f}B")

gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL_NAME).to(device)
gpt2_model.eval() # Freeze GPT-2
for param in gpt2_model.parameters():
    param.requires_grad = False

if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
print(f"GPT-2 모델 로드 완료. 파라미터 수: {sum(p.numel() for p in gpt2_model.parameters()) / 1e9:.3f}B")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[2025-07-03 15:53:36] 모델 및 토크나이저 로드 중...
CLIP 모델 로드 완료. 파라미터 수: 0.428B
GPT-2 모델 로드 완료. 파라미터 수: 1.558B


In [4]:
# --- 3. 매핑 네트워크 정의 ---
class ImageToSoftPromptMapping(nn.Module):
    def __init__(self, clip_embedding_dim, gpt2_hidden_dim, num_soft_tokens):
        super().__init__()
        self.num_soft_tokens = num_soft_tokens
        self.gpt2_hidden_dim = gpt2_hidden_dim 
        self.mlp = nn.Sequential(
            nn.Linear(clip_embedding_dim, gpt2_hidden_dim * num_soft_tokens // 2),
            nn.ReLU(),
            nn.Dropout(0.1), # 오버피팅 방지를 위해 Dropout 추가 (0.1 ~ 0.5)
            nn.Linear(gpt2_hidden_dim * num_soft_tokens // 2, gpt2_hidden_dim * num_soft_tokens)
        )

    def forward(self, image_features):
        mapped_features = self.mlp(image_features)
        soft_prompt_embeddings = mapped_features.view(-1, self.num_soft_tokens, self.gpt2_hidden_dim)
        return soft_prompt_embeddings

CLIP_EMBEDDING_DIM = clip_model.config.projection_dim
GPT2_HIDDEN_DIM = gpt2_model.config.hidden_size      

mapping_network = ImageToSoftPromptMapping(CLIP_EMBEDDING_DIM, GPT2_HIDDEN_DIM, NUM_SOFT_TOKENS).to(device)

total_mapping_params = sum(p.numel() for p in mapping_network.parameters() if p.requires_grad)
print(f"매핑 네트워크 학습 가능한 파라미터: {total_mapping_params / 1e6:.2f}M")
print(f"전체 모델 총 파라미터: {(sum(p.numel() for p in clip_model.parameters()) + sum(p.numel() for p in gpt2_model.parameters()) + total_mapping_params) / 1e9:.3f}B")


매핑 네트워크 학습 가능한 파라미터: 524.34M
전체 모델 총 파라미터: 2.510B


In [7]:
# --- 4. VMCBench 데이터셋 클래스 정의 ---
class VMCBenchDataset(Dataset):
    def __init__(self, annotation_file, question_file, image_dir, tokenizer, clip_processor, split_type):
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.clip_processor = clip_processor
        self.split_type = split_type 

        with open(annotation_file, 'r', encoding='utf-8') as f:
            self.annotations = json.load(f)["annotations"]
        with open(question_file, 'r', encoding='utf-8') as f:
            self.questions_data = {q["question_id"]: q for q in json.load(f)["questions"]}

        self.annotations = [anno for anno in self.annotations if anno["question_id"] in self.questions_data]

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        anno = self.annotations[idx]
        question_id = anno["question_id"]
        image_id = anno["image_id"]
        
        question_info = self.questions_data[question_id]
        question_text = question_info["question"] 

        # 이미지 로드
        image_filename = f"COCO_{self.split_type}_{str(image_id).zfill(12)}.jpg"
        image_path = os.path.join(self.image_dir, image_filename)
        try:
            image = Image.open(image_path).convert("RGB")
        except FileNotFoundError:
            print(f"Warning: Image file not found: {image_path}. Returning None for this item.")
            return None 
            
        clip_image_inputs = self.clip_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

        full_prompt = f"{question_text}\nAnswer:"
        gpt2_inputs = self.tokenizer(full_prompt, return_tensors="pt", padding=False, truncation=True, max_length=512)

        target_char_label = chr(ord('A') + anno["answer_label"])
        target_ids = self.tokenizer(target_char_label, return_tensors="pt")["input_ids"].squeeze(0)
        
        labels_for_collate_item = torch.cat([gpt2_inputs["input_ids"].squeeze(0), target_ids], dim=0)

        return {
            "clip_image_inputs": clip_image_inputs,
            "gpt2_input_ids_for_collate": gpt2_inputs["input_ids"].squeeze(0),
            "attention_mask_for_collate": gpt2_inputs["attention_mask"].squeeze(0),
            "target_char_label": target_char_label, # 정확도 계산을 위해 실제 정답 문자도 반환
            "target_token_id": target_ids.item() # 단일 스칼라 값으로 반환
        }

In [8]:
# --- 5. Custom Collate Function ---
def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch:
        return None 
    
    clip_image_inputs = torch.stack([item['clip_image_inputs'] for item in batch])
    
    max_len_text_input = max(item['gpt2_input_ids_for_collate'].size(0) for item in batch)
    
    max_seq_len_for_inputs_and_labels = NUM_SOFT_TOKENS + max_len_text_input 

    padded_gpt2_input_ids = []
    padded_attention_mask = []
    padded_labels = []
    target_char_labels_batch = [] 

    for item in batch:
        gpt2_input_ids_unpadded = item['gpt2_input_ids_for_collate']
        attention_mask_unpadded = item['attention_mask_for_collate']
        target_token_id = item['target_token_id'] 
        target_char_labels_batch.append(item['target_char_label'])
        
        pad_len_input = max_len_text_input - gpt2_input_ids_unpadded.size(0)
        padded_gpt2_input_ids_item = torch.cat([gpt2_input_ids_unpadded, torch.full((pad_len_input,), gpt2_tokenizer.pad_token_id, dtype=torch.long)], dim=0)
        padded_gpt2_input_ids.append(padded_gpt2_input_ids_item)
        
        padded_attention_mask_item = torch.cat([attention_mask_unpadded, torch.zeros(pad_len_input, dtype=torch.long)], dim=0)
        padded_attention_mask.append(padded_attention_mask_item)
        
        current_item_labels = torch.full((max_seq_len_for_inputs_and_labels,), -100, dtype=torch.long)
        
        answer_label_position = NUM_SOFT_TOKENS + gpt2_input_ids_unpadded.size(0) - 1

        if answer_label_position >= 0 and \
           answer_label_position < max_seq_len_for_inputs_and_labels:
            current_item_labels[answer_label_position] = target_token_id
        else:
            print(f"CRITICAL WARNING: Answer label position({answer_label_position}) out of bounds for labels length({max_seq_len_for_inputs_and_labels}). This item's label might be incorrect. GPT2 input len: {gpt2_input_ids_unpadded.size(0)}, Soft tokens: {NUM_SOFT_TOKENS}, Max Text Input: {max_len_text_input}, Target ID: {target_token_id}")

        padded_labels.append(current_item_labels)

    return {
        "clip_image_inputs": clip_image_inputs,
        "gpt2_input_ids": torch.stack(padded_gpt2_input_ids),
        "attention_mask": torch.stack(padded_attention_mask),
        "labels": torch.stack(padded_labels),
        "target_char_labels": target_char_labels_batch 
    }


In [12]:
# --- 6. DataLoader Creation ---
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 데이터로더 생성 중...")

# **여기에 train_dataset 및 val_dataset 인스턴스 생성 코드가 위치해야 합니다.**

# train_dataset 인스턴스 생성 (이 부분이 누락되었거나 실행되지 않았을 수 있습니다)
train_dataset = VMCBenchDataset(
    annotation_file=TRAIN_ANNOTATION_FILE,
    question_file=TRAIN_QUESTION_FILE,
    image_dir=TRAIN_IMAGE_DIR,
    tokenizer=gpt2_tokenizer,
    clip_processor=clip_processor,
    split_type="train2014"
)
print(f"훈련 데이터셋 크기: {len(train_dataset)}")

# val_dataset 인스턴스 생성
val_dataset = VMCBenchDataset(
    annotation_file=VAL_ANNOTATION_FILE,
    question_file=VAL_QUESTION_FILE,
    image_dir=VAL_IMAGE_DIR,
    tokenizer=gpt2_tokenizer,
    clip_processor=clip_processor,
    split_type="val2014"
)
print(f"검증 데이터셋 크기: {len(val_dataset)}")


# DataLoader 생성 시 generator를 사용하여 셔플의 재현성을 확보합니다.
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
                              generator=torch.Generator().manual_seed(SEED)) 
print(f"훈련 데이터로더 배치 수: {len(train_dataloader)}")

val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
                            generator=torch.Generator().manual_seed(SEED)) # 검증 데이터는 셔플하지 않지만, 일관성 유지
print(f"검증 데이터로더 배치 수: {len(val_dataloader)}")

[2025-07-03 15:57:01] 데이터로더 생성 중...
훈련 데이터셋 크기: 1000
검증 데이터셋 크기: 60
훈련 데이터로더 배치 수: 500
검증 데이터로더 배치 수: 30


In [13]:
# --- 7. Training Loop ---
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 매핑 네트워크 학습 시작...")
optimizer = torch.optim.AdamW(mapping_network.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler() 

best_val_loss = float('inf')
for epoch in range(NUM_EPOCHS):
    mapping_network.train()
    total_train_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} (Train)")

    for batch in progress_bar:
        if batch is None:
            continue

        optimizer.zero_grad()

        clip_image_inputs = batch["clip_image_inputs"].to(device)
        gpt2_input_ids = batch["gpt2_input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        with torch.no_grad():
            image_features = clip_model.get_image_features(pixel_values=clip_image_inputs)
        
        soft_prompt_embeddings = mapping_network(image_features)

        input_embeddings = gpt2_model.get_input_embeddings()(gpt2_input_ids)
        extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
        
        soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
        extended_attention_mask = torch.cat((soft_prompt_attention_mask, attention_mask), dim=1)
        
        with torch.cuda.amp.autocast():
            outputs = gpt2_model(
                inputs_embeds=extended_input_embeddings,
                attention_mask=extended_attention_mask,
                labels=labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_train_loss += loss.item()
        progress_bar.set_postfix(loss=total_train_loss / (progress_bar.n + 1))

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.4f}")

    # --- 검증 루프 (Validation Loop) ---
    mapping_network.eval()
    total_val_loss = 0
    correct_predictions = 0
    total_predictions = 0
    val_progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} (Validation)")
    with torch.no_grad():
        for batch in val_progress_bar:
            if batch is None:
                continue

            clip_image_inputs = batch["clip_image_inputs"].to(device)
            gpt2_input_ids = batch["gpt2_input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            target_char_labels_batch = batch["target_char_labels"]

            image_features = clip_model.get_image_features(pixel_values=clip_image_inputs)
            soft_prompt_embeddings = mapping_network(image_features)
            input_embeddings = gpt2_model.get_input_embeddings()(gpt2_input_ids)
            extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
            
            soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
            extended_attention_mask = torch.cat((soft_prompt_attention_mask, attention_mask), dim=1)

            # 손실 계산을 위한 GPT-2 포워드 패스
            outputs = gpt2_model(
                inputs_embeds=extended_input_embeddings,
                attention_mask=extended_attention_mask,
                labels=labels
            )
            loss = outputs.loss
            total_val_loss += loss.item()

            # 정확도 계산을 위한 GPT-2 생성 (predict_answer 함수와 유사)
            generated_output_ids = gpt2_model.generate(
                inputs_embeds=extended_input_embeddings,
                attention_mask=extended_attention_mask,
                max_new_tokens=1,
                num_beams=1,
                do_sample=False,
                pad_token_id=gpt2_tokenizer.pad_token_id,
                eos_token_id=gpt2_tokenizer.eos_token_id
            )
            
            # 생성된 토큰 ID 추출 및 디코딩
            generated_token_ids = generated_output_ids[:, extended_input_embeddings.shape[1]].cpu().tolist()
            predicted_chars_batch = [gpt2_tokenizer.decode(tid, skip_special_tokens=True).strip() for tid in generated_token_ids]

            # 정확도 계산
            for i in range(len(predicted_chars_batch)):
                predicted_char = predicted_chars_batch[i]
                true_char = target_char_labels_batch[i]

                if predicted_char not in ['A', 'B', 'C', 'D']:
                    pass 
                elif predicted_char == true_char:
                    correct_predictions += 1
                total_predictions += 1
            
            val_progress_bar.set_postfix(loss=total_val_loss / (val_progress_bar.n + 1), 
                                         accuracy=f"{correct_predictions}/{total_predictions}")

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Epoch {epoch+1} Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        mapping_network_save_path = "best_mapping_network_weights.pth"
        torch.save(mapping_network.state_dict(), mapping_network_save_path)
        print(f"검증 손실 개선! 모델 저장됨: {mapping_network_save_path}")

print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 매핑 네트워크 학습 완료.")

/tmp/ipykernel_3601146/301225990.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[2025-07-03 15:57:39] 매핑 네트워크 학습 시작...


Epoch 1/5 (Train):   0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipykernel_3601146/301225990.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
Epoch 1/5 (Train): 100%|██████████| 500/500 [01:27<00:00,  5.70it/s, loss=2.18]


Epoch 1 Train Loss: 2.1801


Epoch 1/5 (Validation):   0%|          | 0/30 [00:00<?, ?it/s]


IndexError: index 69 is out of bounds for dimension 1 with size 1

In [ ]:
# --- 8. 추론 함수 (학습된 모델 사용) ---
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 추론 예시 시작...")

loaded_mapping_network = ImageToSoftPromptMapping(CLIP_EMBEDDING_DIM, GPT2_HIDDEN_DIM, NUM_SOFT_TOKENS).to(device)
loaded_mapping_network.load_state_dict(torch.load("best_mapping_network_weights.pth"))
loaded_mapping_network.eval()

def predict_answer(image_path, question, choices):
    image = Image.open(image_path).convert("RGB")
    clip_inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_features = clip_model.get_image_features(pixel_values=clip_inputs["pixel_values"])
    
    with torch.no_grad():
        soft_prompt_embeddings = loaded_mapping_network(image_features)

    choices_text = [
        f"A. {choices[0]}",
        f"B. {choices[1]}",
        f"C. {choices[2]}",
        f"D. {choices[3]}"
    ]
    
    joined_choices_text = '\n'.join(choices_text)
    full_question_text = f"{question}\n{joined_choices_text}\nAnswer:"

    gpt2_text_inputs = gpt2_tokenizer(full_question_text, return_tensors="pt").to(device)

    input_embeddings = gpt2_model.get_input_embeddings()(gpt2_text_inputs["input_ids"])

    extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
    
    soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
    extended_attention_mask = torch.cat((soft_prompt_attention_mask, gpt2_text_inputs["attention_mask"]), dim=1)

    with torch.no_grad():
        output_ids = gpt2_model.generate(
            inputs_embeds=extended_input_embeddings,
            attention_mask=extended_attention_mask,
            max_new_tokens=1,
            num_beams=1,
            do_sample=False,
            pad_token_id=gpt2_tokenizer.pad_token_id,
            eos_token_id=gpt2_tokenizer.eos_token_id
        )
    
    # 생성된 토큰 ID 추출 및 디코딩 (IndexError 해결을 위해 output_ids[0, -1] 사용)
    generated_token_id = output_ids[0, -1] 
    predicted_char = gpt2_tokenizer.decode(generated_token_id, skip_special_tokens=True).strip()

    if predicted_char not in ['A', 'B', 'C', 'D']:
        print(f"경고: 예상치 못한 답변 생성: '{predicted_char}'. 랜덤 선택으로 대체합니다.")
        return random.choice(['A', 'B', 'C', 'D'])

    return predicted_char

# --- 9. 대회 테스트 데이터셋으로 추론 ---
TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"
TEST_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images"

df_test_input = pd.read_csv(TEST_CSV_PATH)
df_submission = pd.DataFrame(columns=["ID", "answer"])

print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 대회 테스트 데이터셋으로 추론 시작...")
for index, row in tqdm(df_test_input.iterrows(), total=len(df_test_input), desc="Generating predictions"):
    img_id = row['ID']
    img_path = os.path.join(TEST_IMAGE_DIR, os.path.basename(row['img_path']))
    question = row['Question']
    choices = [row['A'], row['B'], row['C'], row['D']]

    predicted_answer = predict_answer(img_path, question, choices)
    
    df_submission = pd.concat([df_submission, pd.DataFrame([{"ID": img_id, "answer": predicted_answer}])], ignore_index=True)

SUBMISSION_FILE_PATH = "submission.csv"
df_submission.to_csv(SUBMISSION_FILE_PATH, index=False)
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 최종 제출 파일 저장 완료: {SUBMISSION_FILE_PATH}")

[2025-07-03 15:53:48] 데이터로더 생성 중...


NameError: name 'train_dataset' is not defined

Using device: cuda


A new version of the following files was downloaded from https://huggingface.co/vikhyatk/moondream1:
- configuration_moondream.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/vikhyatk/moondream1:
- modeling_phi.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/vikhyatk/moondream1:
- vision_encoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/vikhyatk/moondream1:
- moondream.py
- modeling_phi.py
- vision_encoder.py
. Make sure to double-chec

AttributeError: 'VisionTransformer' object has no attribute '_initialize_weights'